# Project 79 — Classification Fine-Tune Readiness Audit
## Dataset Quality Gates Before Investing in Training

**Stack:** LangChain · Ollama · Pydantic · pandas · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama pydantic pandas

## Step 1 — Sample Classification Dataset

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field
import json, pandas as pd

llm = ChatOllama(model="qwen3:8b", temperature=0.0)

dataset = [
    {"text": "Server is down! Production outage!", "label": "urgent"},
    {"text": "URGENT: payment system failure", "label": "urgent"},
    {"text": "Everything is broken!!!!", "label": "urgent"},
    {"text": "When do you close today?", "label": "inquiry"},
    {"text": "What time do you open?", "label": "inquiry"},
    {"text": "Do you have a student discount?", "label": "inquiry"},
    {"text": "I want to cancel my subscription", "label": "churn"},
    {"text": "Cancel immediately, terrible service", "label": "churn"},
    {"text": "The app crashed again", "label": "bug"},
    {"text": "Login button not working on mobile", "label": "bug"},
    {"text": "Love your new feature!", "label": "feedback"},
    {"text": "Nice update, the UI looks great", "label": "feedback"},
    {"text": "Everything broke after the latest update", "label": "urgent"},  # or bug?
    {"text": "How much does the premium plan cost?", "label": "inquiry"},
    {"text": "Great customer service!", "label": "feedback"},
]

df = pd.DataFrame(dataset)
print(f"Dataset: {len(df)} samples")
print(f"Labels: {df['label'].value_counts().to_dict()}")

## Step 2 — Statistical Quality Checks

In [ ]:
# Check 1: Class balance
counts = df["label"].value_counts()
min_count = counts.min()
max_count = counts.max()
balance_ratio = min_count / max_count
print("CHECK 1: Class Balance")
print(f"  Counts: {counts.to_dict()}")
print(f"  Balance ratio: {balance_ratio:.2f} (target > 0.5)")
print(f"  {'✓ PASS' if balance_ratio > 0.3 else '✗ FAIL'}")

# Check 2: Minimum samples per class
min_threshold = 5
print(f"\nCHECK 2: Minimum Samples (threshold={min_threshold})")
for label, count in counts.items():
    print(f"  {label}: {count} {'✓' if count >= min_threshold else '✗ INSUFFICIENT'}")

# Check 3: Text length distribution
df["text_len"] = df["text"].str.len()
print(f"\nCHECK 3: Text Length")
print(f"  Mean: {df['text_len'].mean():.0f} chars")
print(f"  Min:  {df['text_len'].min()} | Max: {df['text_len'].max()}")
print(f"  Very short (<10): {(df['text_len'] < 10).sum()}")

# Check 4: Total dataset size
print(f"\nCHECK 4: Dataset Size")
print(f"  Total: {len(df)} (recommended: >100 for fine-tuning)")
print(f"  {'✓ OK' if len(df) > 50 else '⚠ Too small for reliable fine-tuning'}")

## Step 3 — LLM-Powered Ambiguity Detection

In [ ]:
class AmbiguityCheck(BaseModel):
    sample_id: int
    assigned_label: str
    could_be: list[str] = Field(description="Other plausible labels")
    ambiguity_score: float = Field(ge=0, le=1)
    reasoning: str

class ReadinessReport(BaseModel):
    readiness: str = Field(description="ready, needs_work, not_ready")
    overall_score: float = Field(ge=0, le=1)
    ambiguous_samples: list[AmbiguityCheck]
    recommendations: list[str]
    estimated_accuracy_ceiling: float = Field(ge=0, le=1)

auditor = llm.with_structured_output(ReadinessReport)

report = auditor.invoke(
    f"Audit this text classification dataset for fine-tuning readiness.\n"
    f"Labels: {list(counts.index)}\n\n{json.dumps(dataset, indent=2)}\n\n"
    f"Check for: ambiguous labels, overlapping categories, inconsistent labeling."
)

print("READINESS AUDIT")
print("=" * 50)
print(f"Readiness: {report.readiness.upper()}")
print(f"Overall score: {report.overall_score:.0%}")
print(f"Estimated accuracy ceiling: {report.estimated_accuracy_ceiling:.0%}")

print(f"\nAmbiguous samples:")
for a in report.ambiguous_samples:
    print(f"  #{a.sample_id} [{a.assigned_label}] could be {a.could_be} (ambiguity={a.ambiguity_score:.0%})")

## Step 4 — Recommendations

In [ ]:
print("RECOMMENDATIONS")
print("=" * 50)
for i, rec in enumerate(report.recommendations, 1):
    print(f"  {i}. {rec}")

# Generate action items
action_prompt = ChatPromptTemplate.from_messages([
    ("system", "Create a prioritized action plan to make this dataset ready for fine-tuning."),
    ("human", "Current state: {len} samples, {labels} labels, readiness={readiness}, "
     "score={score:.0%}, issues: {issues}")
])
action_chain = action_prompt | llm | StrOutputParser()

action_plan = action_chain.invoke({
    "len": len(df),
    "labels": len(counts),
    "readiness": report.readiness,
    "score": report.overall_score,
    "issues": "; ".join(report.recommendations[:3]),
})
print(f"\nACTION PLAN:")
print(action_plan[:500])

## What You Learned
- **Statistical quality gates** for training data
- **Ambiguity detection** with LLM audit
- **Readiness scoring** before fine-tuning investment
- **Actionable recommendations** for dataset improvement